# Generate Shelter Level Synthetic Housing and Homelessness Administrative (SHHAd) Data

```
Program to generate synthetic housing and homelessness system of care data.
Copyright (C) 2026 Geoffrey Messier

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see <https://www.gnu.org/licenses/>. 
```

In [1]:
%load_ext autoreload
%autoreload 1

In [2]:
import numpy as np
import pandas as pd
import re, copy
import pickle

from tqdm.auto import tqdm, trange
from tqdm.notebook import tqdm
tqdm.pandas()

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl

from scipy.special import gamma

In [3]:
COLOR = 'black'
mpl.rcParams['text.color'] = COLOR
mpl.rcParams['axes.labelcolor'] = COLOR
mpl.rcParams['xtick.color'] = COLOR
mpl.rcParams['ytick.color'] = COLOR

In [4]:
procDirStr = './data/'

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 500)
np.set_printoptions(threshold=200)

In [6]:
def pctSumStr(a,b):
    return f'{a:,} / {b:,} ({100*a/b:.1f}%)'

In [7]:
def print_p_matrix(p):
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            print(f"{p[i,j]:10.3g}",end='')
        print('')

In [8]:
def dump_pickle(x,fileStr):
    with open(fileStr, 'wb') as f:
        pickle.dump(x, f, pickle.HIGHEST_PROTOCOL)        
        
def read_pickle(fileStr):
    with open(fileStr, 'rb') as f:
        return pickle.load(f)

## Load Models & System Episodes
---

In [9]:
sysDataAll = read_pickle(procDirStr + 'SHHAd-System-Level-2026_11_03.pkl')

In [10]:
startYear = sysDataAll['StartYear']
nDays = sysDataAll['RecEpoch']
sysRecs = sysDataAll['Records']

In [11]:
shltMrkvAll = read_pickle( procDirStr + 'SHHAd-Shelter-Level-MarkovParameters.pkl' )

pState = shltMrkvAll['pState']
pObs = shltMrkvAll['pObs']
pEnter = shltMrkvAll['pEnter']
stateLbl = shltMrkvAll['stateLbl']
obsLbl = shltMrkvAll['obsLbl']
pctlThrsh = shltMrkvAll['pctlThrsh']
pctlFrac = shltMrkvAll['pctlFrac']

In [12]:
stateNum = dict( zip( list(stateLbl.values()), list(stateLbl.keys()) ) )

In [13]:
tblKeyStat = pd.read_parquet(procDirStr + 'SHHAd-Shelter-Level-KeywordStatistics.parquet')

In [14]:
np.random.seed(121212)

In [15]:
obsLbl

{0: '(Sleep)',
 1: '(Cnslr)',
 2: '(Log)',
 3: '(Sleep,Cnslr)',
 4: '(Sleep,Log)',
 5: '(Cnslr,Log)',
 6: '(Sleep,Cnslr,Log)'}

## Determine Shelter Episode Percentiles
---

In [16]:
nEpisMax = 1000000
epiLength = np.zeros(nEpisMax)

In [17]:
iL = 0
for iR in range( len(sysRecs) ):
    for iE in range( len( sysRecs[iR]['ShelterEpis']['ProgramId'] ) ):
        epiLength[iL] = ( sysRecs[iR]['ShelterEpis']['EndDate'][iE] - sysRecs[iR]['ShelterEpis']['StartDate'][iE] ).days+1
        iL += 1
        

In [18]:
epiLength = pd.Series(epiLength[:iL])

In [19]:
epiLength = epiLength.sort_values()

In [20]:
pctlFrac = [ 0, 0.5, 0.75, 1 ]

In [21]:
print('Group1,Group2,Group3')

pctlIds = {}
pctlThrsh = {}

statKeys = [ 'mean', '50%', 'max', 'count' ]
statStr = { 'mean': 'Mean', '50%': 'Median', 'max': 'Max', 'count': 'n' }

vals = { statStr[k]: [] for k in statKeys }
vals['Lower Group Percentile'] = []
vals['Upper Group Percentile'] = []

for i in range(1,len(pctlFrac)):
    loIdx = int( len(epiLength) * pctlFrac[i-1] )
    hiIdx = int( len(epiLength) * pctlFrac[i] )
    vals['Lower Group Percentile'] += [ f'{pctlFrac[i-1]*100:.0f}th' ]
    vals['Upper Group Percentile'] += [ f'{pctlFrac[i]*100:.0f}th' ]

    statVal = epiLength.iloc[loIdx:hiIdx].describe()
    for k in statKeys:
        if k == 'count':
            vals[statStr[k]] += [ f'{statVal[k]:,.0f}' ]
        else:        
            vals[statStr[k]] += [ f'{statVal[k]:.1f}' ]

    pctlIds[f'Grp{i}'] = epiLength.iloc[loIdx:hiIdx].index.to_numpy() 
    pctlThrsh[f'Grp{i}'] = statVal['max'] 

for k in vals.keys():
    outStr = f'{k} '
    for i in range(len(pctlFrac)-1):
        outStr += f'& {vals[k][i]} '
    print(outStr+'\\\\')
    

Group1,Group2,Group3
Mean & 23.2 & 76.9 & 207.5 \\
Median & 22.0 & 75.0 & 173.0 \\
Max & 52.0 & 109.0 & 1281.0 \\
n & 15,274 & 7,637 & 7,637 \\
Lower Group Percentile & 0th & 50th & 75th \\
Upper Group Percentile & 50th & 75th & 100th \\


## Create Shelter Access Timelines
---

In [22]:
# These CDFs are more convenient for determining state transitions.
pEnterCdf = {}
pStateCdf = {}
pObsCdf = {}
nState = len(stateLbl)
nObs = len(obsLbl)

for k in pEnter.keys():
    pEnterCdf[k] = pEnter[k].cumsum()
    pStateCdf[k] = np.zeros((nState,nState))
    pObsCdf[k] = np.zeros((nState,nObs))    
    for iS in range(nState):
        pStateCdf[k][iS,:] = pState[k][iS,:].cumsum()
        pObsCdf[k][iS,:] = pObs[k][iS,:].cumsum()

In [23]:
# Creates a single random logfile consisting of keyword counts that match DI statistics.
def gen_log_keywords(state):
    logStr = ''
    totWords = 0

    while(totWords == 0):
        for keywordCat in tblKeyStat[state].index:
            nWords = int( np.random.exponential(scale=tblKeyStat.loc[keywordCat,state]) )
            totWords += nWords
        
            # Make sure overdose is mentioned at least once if in an overdose state.
            if state == 'Overdose':
                nWords = max(nWords,1)
        
            if nWords > 0:
                logStr += f'{keywordCat}({nWords}),'
            
    return logStr[:-1]

In [24]:
pctlThrshVals = np.array(list(pctlThrsh.values()))
pctlThrshKeys = np.array(list(pctlThrsh.keys()))

In [25]:
# Generate a single shelter acess entry.
def gen_shelter_entry(state,iR,epiGrp,date):
    row = {}
    row['PersonId'] = iR
    row['Date'] = date
    row['FirstName'] = sysRecs[iR]['FirstName']
    row['LastName'] = sysRecs[iR]['LastName']
    row['Dob'] = sysRecs[iR]['Dob']
    row['ProviderId'] = sysRecs[iR]['ShelterEpis']['ProgramId'][iE] 
    
    u = np.random.uniform()    
    obs = np.searchsorted(pObsCdf[epiGrp][state], u)    
    
    row['ShelterSleep'] = 'Sleep' in obsLbl[obs]
    row['ShelterCounsellor'] = 'Cnslr' in obsLbl[obs]

    if 'Log' in obsLbl[obs]:
        row['ShelterLog'] = gen_log_keywords( stateLbl[state] )

    else:
        row['ShelterLog'] = ''

    return row


In [26]:
tblList = []
nEpiIters = 3 # Number of times to generate a shelter timeline to try and match desired episode length.
endDayErrs = []  # Error between shelter timline length and desired episode length.

for iR in tqdm(range(len(sysRecs))):
    
    nEpis = len(sysRecs[iR]['ShelterEpis']['ProgramId'])

    endDayErrEval = np.zeros(nEpiIters)
    tblEpiEval = [[] for _ in range(nEpiIters)]
    iEval = 0

    for iIter in range(nEpiIters):
        
        endDayErr = 0  # Total disagreement between desired length of all episodes and random timeline durations.
        
        for iE in range(nEpis):
            # Determine the episode stat group based on episode duration.
            epiDays = ( sysRecs[iR]['ShelterEpis']['EndDate'][iE] - sysRecs[iR]['ShelterEpis']['StartDate'][iE] ).days + 1
            epiGrp = pctlThrshKeys[ np.searchsorted(pctlThrshVals,epiDays) ]

            # Init state and date for start of timeline.
            startDate = sysRecs[iR]['ShelterEpis']['StartDate'][iE]
            u = np.random.uniform()
            state = np.searchsorted( pEnterCdf[epiGrp], u ) 

            # Create an entry for each day in the episode.
            for iDay in range(epiDays):
                # Actual entries only created if person is not absent.
                if state != stateNum['Absent']:
                    tblEpiEval[iIter].append( gen_shelter_entry(state,iR,epiGrp,startDate + pd.Timedelta(f'{iDay} days')) )
                    endDate = tblEpiEval[iIter][-1]['Date']

                # Transition to next state.
                u = np.random.uniform()
                state = np.searchsorted( pStateCdf[epiGrp][state], u ) 

            # Determine error between timeline length and desired episode.
            genDays = (endDate - startDate).days + 1
            endDayErr += epiDays - genDays

        endDayErrEval[iIter] = endDayErr

    # Choose the timeline that matches the episode length the best.
    bestIter = np.argmin(endDayErrEval)
    endDayErrs += [ endDayErrEval[bestIter] ]     

    tblList += tblEpiEval[bestIter]


  0%|          | 0/12093 [00:00<?, ?it/s]

In [27]:
errStats = pd.Series(endDayErrs).describe()
print(f'Episode error for {errStats['count']:,.0f} people.')
print(f' Mean: {errStats['mean']:.3f}, Median: {errStats['50%']:.0f}, Max: {errStats['max']:.0f}')

Episode error for 12,093 people.
 Mean: 0.919, Median: 0, Max: 36


## Finalize Table
---

In [28]:
tbl = pd.DataFrame(tblList)

In [29]:
shelterNames = pd.read_csv(procDirStr + 'Names-EmergencyShelters.csv')['Name']
prgIdOffset = 2000

In [30]:
tbl['ServiceType'] = 'Shelter'

In [31]:
tbl = tbl.drop_duplicates(subset=['PersonId','Date'], keep='first') # Some episodes overlap by 1 day.
tbl['StartDate'] = tbl.Date
tbl['EndDate'] = tbl.Date
tbl = tbl.drop(columns=['Date'])

In [32]:
tbl['ProviderName'] = shelterNames[ tbl.ProviderId ].to_numpy()

## Save Table
---

In [34]:
tbl.to_parquet( procDirStr + 'SHHAd-Shelter-Level-2026_11_03.parquet')
tbl.to_csv( procDirStr + f'SHHAd-Shelter-Level-2026_11_03.csv' )